# Notebook 2: Model Training & Evaluation
## When Do Models Fail? Discovering Hidden Failure Patterns

This notebook covers:
1. Temporal train/test split
2. Training Linear Regression, Random Forest, and LSTM
3. Model comparison (MAE, RMSE, MAPE)
4. Prediction analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_ingestion import load_config
from src.models import run_all_models, save_results, temporal_train_test_split

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

config = load_config('../configs/config.yaml')
print('Configuration loaded.')

## 1. Load Engineered Features

In [ ]:
features_df = pd.read_parquet(config['data']['features_file'])
features_df['time_window'] = pd.to_datetime(features_df['time_window'])
print(f'Features shape: {features_df.shape}')
print(f'Zones: {features_df["zone_id"].nunique()}')
features_df.head()

## 2. Train All Models

This step trains:
- **Linear Regression** — simple baseline
- **Random Forest** — captures non-linear patterns
- **LSTM** — learns temporal sequences

Uses temporal split (last 20% = test set) to avoid data leakage.

> **Tip**: For faster experimentation, pass `sample_zones` to limit the number of zones.
> Example: `run_all_models(config, sample_zones=list(range(1, 51)))`

In [ ]:
# Train all models (this may take a while on full data)
# Uncomment the line below to sample 50 zones for faster execution:
# results = run_all_models(config, sample_zones=list(range(1, 51)))

results = run_all_models(config)

## 3. Model Comparison

In [ ]:
# Display metrics comparison
metrics_df = pd.DataFrame(results['metrics'])
print('Model Performance Comparison:')
metrics_df

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

colors = ['#1a73e8', '#34a853', '#f9ab00']
for idx, metric in enumerate(['MAE', 'RMSE', 'MAPE']):
    bars = axes[idx].bar(metrics_df['model'], metrics_df[metric], 
                         color=colors[:len(metrics_df)], alpha=0.85)
    axes[idx].set_title(metric, fontsize=12)
    axes[idx].set_ylabel(metric)
    for bar, val in zip(bars, metrics_df[metric]):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                      f'{val:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: True vs Predicted
test_df = results['test_df']
target = config['features']['target']

n_models = len(results['predictions'])
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
if n_models == 1:
    axes = [axes]

for idx, (model_name, preds) in enumerate(results['predictions'].items()):
    ax = axes[idx]
    if model_name == 'lstm':
        continue  # LSTM has different alignment
    
    y_true = test_df[target].values
    
    # Sample for plotting
    n_sample = min(10000, len(y_true))
    idx_sample = np.random.RandomState(42).choice(len(y_true), n_sample, replace=False)
    
    ax.scatter(y_true[idx_sample], preds[idx_sample], alpha=0.1, s=5)
    max_val = max(y_true[idx_sample].max(), preds[idx_sample].max())
    ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.8, linewidth=1.5)
    ax.set_xlabel('True Demand')
    ax.set_ylabel('Predicted Demand')
    ax.set_title(model_name.replace('_', ' ').title())
    ax.set_xlim(0, min(max_val * 1.1, 500))
    ax.set_ylim(0, min(max_val * 1.1, 500))

plt.suptitle('True vs Predicted Demand', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Save Results

In [ ]:
test_df_saved = save_results(results, config)
print('Results saved!')
print(f'  Models: {config["output"]["models_dir"]}/')
print(f'  Predictions: {config["data"]["processed_dir"]}/predictions.parquet')
print(f'  Metrics: {config["output"]["reports_dir"]}/model_metrics.csv')